---
## ① 加载数据

In [97]:
import pandas as pd

df = pd.read_csv('data/customer_shopping_behaviour_cn.csv')

print(f' 加载完成')
print(f'行数: {len(df):,}')
print(f'列数: {len(df.columns)}')
print(f'\n列名: {list(df.columns)}')

 加载完成
行数: 3,900
列数: 18

列名: ['客户ID', '年龄', '性别', '购买商品', '商品类别', '消费金额(美元)', '所在州', '尺码', '颜色', '季节', '评分', '是否订阅', '配送方式', '是否使用折扣', '是否使用优惠码', '历史购买次数', '支付方式', '购买频率']


In [99]:
df.head()

,客户ID,年龄,性别,购买商品,商品类别,消费金额(美元),所在州,尺码,颜色,季节,评分,是否订阅,配送方式,是否使用折扣,是否使用优惠码,历史购买次数,支付方式,购买频率
0,1,55,男,女式衬衫,服装,53,Kentucky,L,灰色,冬,3.1,是,快递,是,Yes,14,Venmo,每两周
1,2,19,男,毛衣,服装,64,Maine,L,栗色,冬,3.1,是,快递,是,Yes,2,现金,每两周
2,3,50,男,牛仔裤,服装,73,Massachusetts,S,栗色,春,3.1,是,包邮,是,Yes,23,信用卡,每周
3,4,21,男,凉鞋,鞋履,90,Rhode Island,M,栗色,春,3.5,是,次日达,是,Yes,49,PayPal,每周
4,5,45,男,女式衬衫,服装,49,Oregon,M,绿松石色,春,2.7,是,包邮,是,Yes,31,PayPal,每年


In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   客户ID      3900 non-null   int64  
 1   年龄        3900 non-null   int64  
 2   性别        3900 non-null   object 
 3   购买商品      3900 non-null   object 
 4   商品类别      3900 non-null   object 
 5   消费金额(美元)  3900 non-null   int64  
 6   所在州       3900 non-null   object 
 7   尺码        3900 non-null   object 
 8   颜色        3900 non-null   object 
 9   季节        3900 non-null   object 
 10  评分        3863 non-null   float64
 11  是否订阅      3900 non-null   object 
 12  配送方式      3900 non-null   object 
 13  是否使用折扣    3900 non-null   object 
 14  是否使用优惠码   3900 non-null   object 
 15  历史购买次数    3900 non-null   int64  
 16  支付方式      3900 non-null   object 
 17  购买频率      3900 non-null   object 
dtypes: float64(1), int64(4), object(13)
memory usage: 548.6+ KB


In [ ]:
df.describe(include='all')

---
## ② 数据清洗

In [ ]:
# 检查缺失值
miss = df.isnull().sum()
print('=== 缺失值 ===')
print(miss[miss > 0])

In [103]:
# 用商品类别的评分中位数填充
a = df['评分'].isnull().sum()
df['评分'] = df.groupby('商品类别')['评分'].transform(lambda x: x.fillna(x.median()))
b = df['评分'].isnull().sum()
print(f'评分缺失 {a} → {b}')
print('已填充')

评分缺失 37 → 0
已填充


In [105]:
# 删除冗余列
same = (df['是否使用折扣'] == df['是否使用优惠码']).all()
print(f'折扣列和优惠码列完全一致？ {same}')
df = df.drop('是否使用优惠码', axis=1)
print(f'已删除，剩 {len(df.columns)} 列')

折扣列和优惠码列完全一致？ False
已删除，剩 17 列


---
## ③ 特征工程

In [108]:
# 年龄段（年龄四等分）
df['年龄段'] = pd.qcut(df['年龄'], q=4, labels=['青年', '成年', '中年', '老年'])
print('=== 年龄段分布 ===')
print(df['年龄段'].value_counts())
df[['年龄', '年龄段']].head(8)

=== 年龄段分布 ===
年龄段
青年    1028
中年     986
老年     944
成年     942
Name: count, dtype: int64


,年龄,年龄段
0,55,中年
1,19,青年
2,50,中年
3,21,青年
4,45,中年
5,46,中年
6,63,老年
7,27,青年


In [110]:
# 购买频率 → 天数
freq_map = {
    '每周': 7, '每两周': 14, '双周': 14,
    '每月': 30, '每季': 90, '每三个月': 90, '每年': 365
}
df['购买频率天数'] = df['购买频率'].map(freq_map)
df[['购买频率', '购买频率天数']].head(8)

,购买频率,购买频率天数
0,每两周,14
1,每两周,14
2,每周,7
3,每周,7
4,每年,365
5,每周,7
6,每季,90
7,每周,7


In [112]:
# 客户分层
def fen_ceng(ci):
    if ci == 1:
        return '新客户'
    elif ci <= 10:
        return '回头客'
    else:
        return '忠实客户'

df['客户类型'] = df['历史购买次数'].apply(fen_ceng)
print('=== 客户分层 ===')
print(df['客户类型'].value_counts())
for t in ['新客户', '回头客', '忠实客户']:
    c = len(df[df['客户类型'] == t])
    print(f'  {t}: {c} 人 ({c/len(df)*100:.1f}%)')

=== 客户分层 ===
客户类型
忠实客户    3116
回头客      701
新客户       83
Name: count, dtype: int64
  新客户: 83 人 (2.1%)
  回头客: 701 人 (18.0%)
  忠实客户: 3116 人 (79.9%)


---
## ④ 保存清洗后的数据

In [114]:
df.to_csv('data/清洗后数据.csv', index=False, encoding='utf-8-sig')
print(f'已保存: data/清洗后数据.csv')
print(f'行: {len(df):,} | 列: {len(df.columns)}')

已保存: data/清洗后数据.csv
行: 3,900 | 列: 20


In [119]:

df_sql = df.rename(columns={'消费金额(美元)': '消费金额_美元'})

USER = 'root'
PASSWORD = 'root'        
HOST = 'localhost'
DB = 'customer_behaviour'

from sqlalchemy import create_engine

engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB}?charset=utf8mb4')

df_sql.to_sql('customer', engine, if_exists='replace', index=False)
print(f'已写入 [{DB}].customer')
print(f'行: {len(df_sql):,} | 列: {len(df_sql.columns)}')

pd.read_sql('SELECT * FROM customer LIMIT 3;', engine)

已写入 [customer_behaviour].customer
行: 3,900 | 列: 20


,客户ID,年龄,性别,购买商品,商品类别,消费金额_美元,所在州,尺码,颜色,季节,评分,是否订阅,配送方式,是否使用折扣,历史购买次数,支付方式,购买频率,年龄段,购买频率天数,客户类型
0,1,55,男,女式衬衫,服装,53,Kentucky,L,灰色,冬,3.1,是,快递,是,14,Venmo,每两周,中年,14,忠实客户
1,2,19,男,毛衣,服装,64,Maine,L,栗色,冬,3.1,是,快递,是,2,现金,每两周,青年,14,回头客
2,3,50,男,牛仔裤,服装,73,Massachusetts,S,栗色,春,3.1,是,包邮,是,23,信用卡,每周,中年,7,忠实客户
